In [40]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut, RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

df = pd.read_csv("components.csv")

X = df.drop(columns=["measurement","no","id"]).values
y = df["measurement"].values

model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", Ridge(alpha=1.0))
])

print(X.shape)
print(X[:5])  # Print first 5 rows of X
print(y.shape)
print(y[:5]) # Print first 5 values of y

(18, 14)
[[1.00000000e+02 2.30000000e+01 2.30000000e+01 1.88800000e+01
  5.70000000e+01 3.00000000e+01 2.14000000e+01 3.26000000e+01
  2.90000000e-01 5.75000000e+02 6.03100000e-02 2.47600000e-01
  5.83200000e+00 1.44670746e+00]
 [2.00000000e+02 2.30000000e+01 2.30000000e+01 7.80000000e+01
  5.70000000e+01 3.00000000e+01 2.14000000e+01 3.26000000e+01
  2.00000000e-01 1.15000000e+03 3.65000000e-02 1.40000000e-01
  5.77500000e+00 1.44670746e+00]
 [2.00000000e+02 2.30000000e+01 5.00000000e+01 3.23000000e+01
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.90000000e+01
  2.00000000e-01 8.40000000e+02 5.15300000e-02 2.50800000e-01
  7.12800000e+00 1.42544002e+00]
 [2.00000000e+02 1.00000000e+02 5.00000000e+01 3.24000000e+01
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.90000000e+01
  2.90000000e-01 8.40000000e+02 5.69100000e-02 2.48140000e-01
  7.12800000e+00 1.42544002e+00]
 [1.00000000e+02 2.30000000e+01 5.00000000e+01 8.32000000e+00
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.9000

In [41]:

# --- LOOCV ---
loo = LeaveOneOut()
preds_loo = []
truth_loo = []

for train_idx, test_idx in loo.split(X):
    model.fit(X[train_idx], y[train_idx])
    preds_loo.append(model.predict(X[test_idx])[0])
    truth_loo.append(y[test_idx][0])

rmse_loo = np.sqrt(mean_squared_error(truth_loo, preds_loo))
print("LOOCV RMSE:", rmse_loo)

# --- Repeated K-Fold ---
rkf = RepeatedKFold(n_splits=4, n_repeats=10, random_state=42)
preds_rkf = []
truth_rkf = []

for train_idx, test_idx in rkf.split(X):
    model.fit(X[train_idx], y[train_idx])
    preds_rkf.extend(model.predict(X[test_idx]))
    truth_rkf.extend(y[test_idx])

rmse_rkf = np.sqrt(mean_squared_error(truth_rkf, preds_rkf))
print("Repeated K-Fold RMSE:", rmse_rkf)


LOOCV RMSE: 4.5423672601497875
Repeated K-Fold RMSE: 5.222914931247047


In [42]:
coef = model.named_steps["reg"].coef_
for name, c in zip(df.columns[:-1].drop(["no","id"]), coef):
    print(name, c)

pri -1.2804598120845205
sec 0.6306866706879617
va 2.138189337361243
dcr 1.3244802451674245
size 2.6709031729542905
depth 1.262091772043144
bobbin_width 2.734078945003464
bobbin_depth 1.3726504983464598
pri_dia 5.2746819185962615
pri_turns 0.11741063674382106
io 5.315519319011211
il 6.058161973570942
coil_height 0.44804277605338166
bmax -0.5803417060180485


In [43]:
df2 = pd.read_csv("predict.csv")

X2 = df2.drop(columns=["no","id"]).values

preds = model.predict(X2)    

print("Predicted inrush currents for new data:")
#for i, pred in enumerate(preds):
#    print(f"Sample {i+1}: {pred:.2f}")

# for easy copy-paste
for i, pred in enumerate(preds):
    print(f"{pred:.2f} ")



Predicted inrush currents for new data:
10.57 
25.88 
44.28 
68.10 
85.01 
113.93 
112.13 
156.86 
186.40 
7.65 
11.74 
26.82 
41.46 
53.12 
69.42 
74.90 
108.78 
129.65 
5.55 
9.93 
24.31 
36.91 
52.59 
67.12 
70.51 
107.37 
124.23 


In [44]:
new_transformer = {
    "pri": 200,
    "sec": 100,
    "va": 50,
    "dcr": 31.58,
    "size": 66,
    "width": 22,
    "bobbin_width": 30,
    "bobbin_depth": 25,
    "pri_dia": 0.29,
    "pri_turns": 930,
    "io": 0.1037,
    "il": 0.3064,
    "coil_height": 2.94,
    "bmax": 1.545
}


In [45]:
x_new = np.array([
    new_transformer["pri"],
    new_transformer["sec"],
    new_transformer["va"],
    new_transformer["dcr"],
    new_transformer["size"],
    new_transformer["width"],
    new_transformer["bobbin_width"],
    new_transformer["bobbin_depth"],
    new_transformer["pri_dia"],
    new_transformer["pri_turns"],
    new_transformer["io"],
    new_transformer["il"],
    new_transformer["coil_height"],
    new_transformer["bmax"],
]).reshape(1, -1)

for i, col in enumerate(df.columns[:-1].drop(["no","id"])):
    v = float(x_new[0, i])
    lo, hi = float(df[col].min()), float(df[col].max())
    if not ((lo <= v) and (v <= hi)):
        print(f"Warning: {col}={v} is outside training range [{lo}, {hi}]")

predicted_inrush = model.predict(x_new)
print("Predicted inrush current:", predicted_inrush[0])


Predicted inrush current: 9.675385930206424


In [46]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.metrics import mean_squared_error

kernel = ConstantKernel(1.0, (1e-4, 1e4)) * \
         RBF(length_scale=1.0, length_scale_bounds=(1e-8, 1e4)) + \
         WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e1))
reg = GaussianProcessRegressor(kernel=kernel)


df = pd.read_csv("components.csv")

X = df.drop(columns=["measurement","no","id"]).values
y = df["measurement"].values


model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", reg)
])


print(X.shape)
print(X[:5])  # Print first 5 rows of X
print(y.shape)
print(y[:5]) # Print first 5 values of y

(18, 14)
[[1.00000000e+02 2.30000000e+01 2.30000000e+01 1.88800000e+01
  5.70000000e+01 3.00000000e+01 2.14000000e+01 3.26000000e+01
  2.90000000e-01 5.75000000e+02 6.03100000e-02 2.47600000e-01
  5.83200000e+00 1.44670746e+00]
 [2.00000000e+02 2.30000000e+01 2.30000000e+01 7.80000000e+01
  5.70000000e+01 3.00000000e+01 2.14000000e+01 3.26000000e+01
  2.00000000e-01 1.15000000e+03 3.65000000e-02 1.40000000e-01
  5.77500000e+00 1.44670746e+00]
 [2.00000000e+02 2.30000000e+01 5.00000000e+01 3.23000000e+01
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.90000000e+01
  2.00000000e-01 8.40000000e+02 5.15300000e-02 2.50800000e-01
  7.12800000e+00 1.42544002e+00]
 [2.00000000e+02 1.00000000e+02 5.00000000e+01 3.24000000e+01
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.90000000e+01
  2.90000000e-01 8.40000000e+02 5.69100000e-02 2.48140000e-01
  7.12800000e+00 1.42544002e+00]
 [1.00000000e+02 2.30000000e+01 5.00000000e+01 8.32000000e+00
  6.60000000e+01 3.60000000e+01 2.47000000e+01 3.9000

In [47]:
# --- LOOCV Evaluation ---
loo = LeaveOneOut()
preds = []
truth = []
stds = []

for train_idx, test_idx in loo.split(X):
    model.fit(X[train_idx], y[train_idx])
    mean, std = model.predict(X[test_idx], return_std=True)
    preds.append(mean[0])
    stds.append(std[0])
    truth.append(y[test_idx][0])

rmse = np.sqrt(mean_squared_error(truth, preds))
print("LOOCV RMSE:", rmse)

# Optional: print average uncertainty
print("Mean predictive std:", np.mean(stds))


LOOCV RMSE: 5.456066692951759
Mean predictive std: 3.813452615648594


In [48]:
x_new = np.array([
    new_transformer["pri"],
    new_transformer["sec"],
    new_transformer["va"],
    new_transformer["dcr"],
    new_transformer["size"],
    new_transformer["width"],
    new_transformer["bobbin_width"],
    new_transformer["bobbin_depth"],
    new_transformer["pri_dia"],
    new_transformer["pri_turns"],
    new_transformer["io"],
    new_transformer["il"],
    new_transformer["coil_height"],
    new_transformer["bmax"],
]).reshape(1, -1)

predicted_inrush = model.predict(x_new)
print("Predicted inrush current:", predicted_inrush)


Predicted inrush current: [23.16744535]
